In [ ]:
!pip install --upgrade scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 71.6 MB/s eta 0:00:00
  Attempting uninstall: scikit-learn
    Found existing installation: scikit-learn 1.6.1
    Uninstalling scikit-learn-1.6.1:
      Successfully uninstalled scikit-learn-1.6.1


In [ ]:
import xgboost as xgb
import shap
import pandas as pd
import numpy as np
from typing import Union, Dict, Optional, Tuple, Set, List
from math import factorial
import time
from copy import copy
from tqdm import tqdm
from collections import defaultdict
from sklearn.metrics import accuracy_score, f1_score
import sklearn
import math

import warnings
warnings.filterwarnings("ignore", category=FutureWarning, module=r"sklearn\..*")

In [ ]:
# Useful if you run this on google colab and downloaded the data into your drive.
# If you run the notebook in other environment remove these lines and change the 'pd.read_csv()' function in this notebook to read from
# where you saved you data
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# import woodelf from Python file in the drive
!cp /content/drive/MyDrive/...../woodelf.py /content/

import woodelf

# PDP Code

In [ ]:
class CPDVMetric(woodelf.CubeMetric):
    def calc_metric(self, s_plus: Set, s_minus: Set) -> Dict[str, float]:
        if len(s_plus & s_minus) > 0:
            return {}
        pdp_values = {}
        if len(s_plus) == 1:
            for f in s_plus:
                pdp_values[f] = 1
        if len(s_plus) == 0:
            for f in s_minus:
                pdp_values[f] = -1
        return pdp_values

In [ ]:
class PathToValuesMatrixLimitSPlus(woodelf.PathToValuesMatrix):
    # Ignored all cubes with |S^+| > MAX_S_PLUS_SIZE to reduce the complexity element of TL3**D to TL2**D*(D**MAX_S_PLUS_SIZE)
    MAX_S_PLUS_SIZE = NotImplemented

    @classmethod
    def map_patterns_to_cube(cls, features_in_path: List[str]):
        updated_wdnf_table = {0: {0: (set(), set())}}
        current_wdnf_table = None
        for feature in features_in_path:
            current_wdnf_table = updated_wdnf_table
            updated_wdnf_table = {}
            for consumer_pattern in current_wdnf_table:
                updated_wdnf_table[consumer_pattern * 2 + 0] = {}
                updated_wdnf_table[consumer_pattern * 2 + 1] = {}
                for background_pattern in current_wdnf_table[consumer_pattern]:
                    s_plus, s_minus = current_wdnf_table[consumer_pattern][background_pattern]

                    # The implementation is identical to the PathToValuesMatrix.map_patterns_to_cube implementation, except for this if.
                    if len(s_plus | {feature}) <= cls.MAX_S_PLUS_SIZE:
                        updated_wdnf_table[consumer_pattern * 2 + 1][background_pattern * 2 + 0] = (s_plus | {feature}, s_minus) # Rule 1

                    updated_wdnf_table[consumer_pattern * 2 + 0][background_pattern * 2 + 1] = (s_plus, s_minus | {feature}) # Rule 2
                    updated_wdnf_table[consumer_pattern * 2 + 1][background_pattern * 2 + 1] = (s_plus, s_minus) # Rule 3

        return updated_wdnf_table

class PathToValuesMatrixLimitSPlusTo1(PathToValuesMatrixLimitSPlus):
    # We uses the fact CPDVMetric ignored all cubes with |S^+| > 1 to reduce the complexity element of TL3**D to TL2**D*D
    MAX_S_PLUS_SIZE = 1

class PathToValuesMatrixLimitSPlusTo2(PathToValuesMatrixLimitSPlus):
    # We uses the fact PDIVOrder1Or2 ignored all cubes with |S^+| > 2 to reduce the complexity element of TL3**D to TL(2**D)*(D**2)
    MAX_S_PLUS_SIZE = 2

In [ ]:
def build_sampled_points_df(data: pd.DataFrame, k: int, seed: int = None):
    """
    Sample k points from every column.
    """
    sample_points_data = {}
    for f in data.columns:
        sample_points_data[f] = list(data[f].sample(k, random_state=seed))
        sample_points_data[f].sort()
    return pd.DataFrame(sample_points_data)[data.columns]

def build_equally_distanced_points_df(data: pd.DataFrame, k: int, percentiles: Tuple[float]):
    """
    Take equally distanced points from each column. The min point will be in the precentile percentiles[0]
    and the max point will be in the precentile percentiles[1].
    This is also the default implementation of sklearn
    """
    sample_points_data = {}
    for f in data.columns:
        low, high = np.percentile(data[f].dropna(), [percentiles[0] * 100, percentiles[1]*100])
        # get k equally spaced points between them
        points = np.linspace(low, high, k)
        sample_points_data[f] = list(points)
        sample_points_data[f].sort()
    return pd.DataFrame(sample_points_data)[data.columns]

def build_points_for_full_pdp(data: pd.DataFrame, model, as_df: bool=True):
    """
    Provide the points that will create a full PDP - a graph the will provide the PDV for every x value.
    Does this by collecting all the threshold values from the model. See Sect. of the paper.
    """
    # load the model
    model_objs = woodelf.load_decision_tree_ensamble_model(model, list(data.columns))

    # collect all the theshold values for each feature
    th_values = {f: [] for f in list(data.columns)}
    for tree in model_objs:
        for node in tree.bfs(including_myself=True, including_leaves=False):
            th_values[node.feature_name].append(node.value)

    # Make sure the thesholds are unique and sort them
    for f in th_values:
        th_values[f] = sorted(list(set(th_values[f])))

    if not as_df:
        return th_values

    # zfill
    max_th_length = max([len(thersholds) for thersholds in th_values.values()])
    for f in th_values:
        th_values[f].extend([0] * (max_th_length - len(th_values[f])) )

    # from the built thershold build the points Data Frame
    return pd.DataFrame(th_values)

In [ ]:
def build_points_for_pdp(model, data: pd.DataFrame, k: int = 100, percentiles: Tuple[float] = (0.05, 0.95), sampled: bool = False, seed: int = 42, full_pdp: bool = False, verbose : bool = True):
    start_time = time.time()
    if sampled:
        points_df = build_sampled_points_df(data, k, seed)
    elif full_pdp:
        points_df = build_points_for_full_pdp(data, model)
    else:
        points_df = build_equally_distanced_points_df(data, k, percentiles)
    if verbose:
        print(f"Building the points took: {time.time() - start_time} sec")
    return points_df

def woodelf_pdp(model, data: pd.DataFrame, k: int = 100, accurate: bool = True, centered: bool = True, GPU: bool = False,
                percentiles: Tuple[float] = (0.05, 0.95), sampled: bool = False, seed: int = 42, full_pdp: bool = False):
    """
    Compute all the PDVs needed in order to plot the PDP values of all the features. Use WOODELF!
    """
    points_df = build_points_for_pdp(model, data, k, percentiles, sampled, seed, full_pdp, verbose=True)
    return woodelf_pdp_given_points_df(model, data, points_df, accurate, centered, GPU), points_df

def woodelf_pdp_given_points_df(model, data: pd.DataFrame, sampled_points_df: pd.DataFrame, accurate: bool = True, centered: bool = True, GPU: bool = False):
    """
    Compute all the PDVs of the provided points. Use WOODELF!
    """
    metric=CPDVMetric()
    p2v = PathToValuesMatrixLimitSPlusTo1(metric)
    if accurate:
        pdvs = woodelf.calculate_background_metric(model, consumer_data=sampled_points_df, background_data=data, metric=metric, global_importance=False, GPU=GPU, path_to_matrixes_calculator=p2v)
    else:
        pdvs = woodelf.calculate_path_dependent_metric(model, consumer_data=sampled_points_df, metric=metric, global_importance=False, GPU=GPU, path_to_matrixes_calculator=p2v)
    if centered:
        return pdvs

    avg_prediction = float(model.predict(data).mean())
    for f in pdvs:
        pdvs[f] += avg_prediction
    return pdvs

In [ ]:
# PDP joint

from itertools import combinations

def all_subsets_of_size_0_1_2(s):
    subsets = [set()]
    for k in [1,2]:
        for subset in combinations(s, k):
            subsets.append(set(subset))
    return subsets

class PDIVOrder1Or2(woodelf.CubeMetric):
    INTERACTION_VALUE = True

    def calc_metric(self, s_plus: Set, s_minus: Set) -> Dict[str, float]:
        if len(s_plus & s_minus) > 0:
            return {}

        pdivs = {}
        for sm in all_subsets_of_size_0_1_2(s_minus):
            s = tuple(s_plus | sm)
            if len(s) in [1,2]:
                pdivs[s] = (-1) ** (len(sm))
        return pdivs

def bits(n, D):
    bs = []
    for i in range(D):
        bs.append(n % 2)
        n = n // 2
    return reversed(bs)

def build_points_for_joint_pdp(points_df: pd.DataFrame):
    D = math.ceil(math.log2(len(points_df.columns)))
    data = {f: [] for f in points_df.columns}
    k = len(points_df)
    for i, f in enumerate(points_df.columns):
        for b in bits(i, D):
            if b == 0:
                data[f].extend(np.tile(points_df[f].values, k))
            elif b == 1:
                data[f].extend(np.repeat(points_df[f].values, k))
    return pd.DataFrame(data)

def first_different_bit(n1, n2, D):
    assert n1 != n2
    i = 0
    for b1, b2 in zip(bits(n1, D), bits(n2, D)):
        if b1 != b2:
            return i
        i += 1

def clip_result(pdvs, features, k):
    D = math.ceil(math.log2(len(features)))
    feature_to_index = {f:i for i,f in enumerate(features)}
    clipped = {}
    for f1, f2 in pdvs:
        i1 = feature_to_index[f1]
        i2 = feature_to_index[f2]
        h = first_different_bit(i1, i2, D)
        clipped[(f1, f2)] = pdvs[(f1, f2)][h*(k**2): (h+1)*(k**2)]
    return clipped


def woodelf_pdp_joint(model, data: pd.DataFrame, k: int = 100, accurate: bool = True, centered: bool = True, GPU: bool = False,
                percentiles: Tuple[float] = (0.05, 0.95), sampled: bool = False, seed: int = 42, full_pdp: bool = False, verbose: bool = True):
    """
    Compute all the PDVs needed in order to plot the PDP values of all the features. Use WOODELF!
    """
    start_time = time.time()
    original_points_df = build_points_for_pdp(model, data, k, percentiles, sampled, seed, full_pdp, verbose=False)
    if full_pdp:
        k = len(original_points_df)

    points_df = build_points_for_joint_pdp(original_points_df)
    if verbose:
        print(f"Building the points took: {time.time() - start_time} sec. The size of the created df {len(points_df)}")

    metric = PDIVOrder1Or2()
    p2v = PathToValuesMatrixLimitSPlusTo2(metric)
    if accurate:
        pdivs = woodelf.calculate_background_metric(
            model, consumer_data=points_df, background_data=data, metric=metric, global_importance=False, GPU=GPU, path_to_matrixes_calculator=p2v
        )
    else:
        pdivs = woodelf.calculate_path_dependent_metric(
            model, consumer_data=points_df, metric=metric, global_importance=False, GPU=GPU, path_to_matrixes_calculator=p2v
        )
    avg_prediction = float(model.predict(data).mean())
    base_pdv = np.array([avg_prediction] * len(points_df))
    zero_array = np.array([0] * len(points_df))
    pdvs = {}

    D = math.ceil(math.log2(len(points_df.columns)))
    points_parts = {f: [points_df[f].values[i:i + k**2] for i in range(0, len(points_df[f]), k**2)] for f in data.columns}
    f1_points = {}
    f2_points = {}
    for i, f1 in enumerate(data.columns):
        for j, f2 in enumerate(data.columns):
            if f1 < f2:
            # if f1 != f2:
                pair = (f1, f2) # if f1 < f2 else (f2, f1)
                pdvs[(f1,f2)] = base_pdv + pdivs.get((f1,), zero_array) + pdivs.get((f2,), zero_array) + pdivs.get(pair, zero_array)
                points_part_index = first_different_bit(i,j,D)
                f1_points[(f1,f2)] = points_parts[f1][points_part_index]
                f2_points[(f1,f2)] = points_parts[f2][points_part_index]
    clipped_pdvs = clip_result(pdvs, list(data.columns), k)
    return clipped_pdvs, f1_points, f2_points

# Fraud Data + Model

In [ ]:
transactions_train = pd.read_parquet('drive/MyDrive/PATH_TO_DATA_CREATED_IN_THE_Preprocess_Data_NOTEBOOK.parquet') # columns are train_features + ['isFraud']
transactions_test = pd.read_parquet('drive/MyDrive/PATH_TO_DATA_CREATED_IN_THE_Preprocess_Data_NOTEBOOK.parquet') # columns are train_features + ['isFraud']

train_features = [f for f in transactions_train.columns if f != 'isFraud']
fraud_train = transactions_train[train_features]
fraud_test = transactions_test[train_features]

In [ ]:
def train_hist_gradient_boosting_model(X, y, max_depth=6):
    gradient_boosting_model = sklearn.ensemble.HistGradientBoostingRegressor(
        max_iter=100,
        max_depth=max_depth,
        max_leaf_nodes=None,
        random_state=42,
        min_samples_leaf=1
    )
    gradient_boosting_model.fit(X, y)
    return gradient_boosting_model

models = {depth: train_hist_gradient_boosting_model(fraud_train, transactions_train['isFraud'], depth) for depth in tqdm(range(1, 11))}

for depth, model in models.items():
    y_pred = model.predict(transactions_test[train_features])
    print(f"Depth {depth}. Accuracy: {accuracy_score(transactions_test['isFraud'], y_pred.round())}, F1 score: {f1_score(transactions_test['isFraud'], y_pred.round())}")

100%|██████████| 10/10 [04:14<00:00, 25.45s/it]


Depth 1. Accuracy: 0.9702645036746029, F1 score: 0.27347952006619775
Depth 2. Accuracy: 0.9706116435804518, F1 score: 0.32875652678398765
Depth 3. Accuracy: 0.9718901344532123, F1 score: 0.37452901281085155
Depth 4. Accuracy: 0.9720594709926508, F1 score: 0.3938280675973549
Depth 5. Accuracy: 0.9729400209977309, F1 score: 0.41721371261852663
Depth 6. Accuracy: 0.9715006604125038, F1 score: 0.4133844545137679
Depth 7. Accuracy: 0.9622802858400785, F1 score: 0.35218845426784934
Depth 8. Accuracy: 0.9724150777254716, F1 score: 0.4347675225537821
Depth 9. Accuracy: 0.9640583195041826, F1 score: 0.3814658312691243
Depth 10. Accuracy: 0.9624919565143767, F1 score: 0.3726989521382045


## Woodelf PDP computation



In [ ]:
def df_to_latex(df: pd.DataFrame):
    latex = "\\begin{tabularx}{\\columnwidth}{" + "|".join(['r'] * len(df.columns)) + "} \n"
    latex += " & ".join(["\\textbf{" + str(c) + "}" for c in df.columns]) + "\\\\\\hline \n"
    for index, row in df.iterrows():
        latex += " & ".join([f" {v} " for v in row]) + " \\\\ \n"
    latex += "\\end{tabularx}"
    print(latex)

def get_pdp_testing_params():
    return [
        ("Exact PDP k=5",                  woodelf_pdp, dict(k = 5,         accurate = True,  centered = False, GPU = False)),
        ("Exact PDP k=100",                woodelf_pdp, dict(k = 100,       accurate = True,  centered = False, GPU = False)),
        ("Exact full PDP",                 woodelf_pdp, dict(full_pdp=True, accurate = True,  centered = False, GPU = False)),
        ("Exact Joint PDP k=5",      woodelf_pdp_joint, dict(k = 5,         accurate = True,  centered = False, GPU = False)),
        ("Estimated PDP k=5",              woodelf_pdp, dict(k = 5,         accurate = True,  centered = False, GPU = False)),
        ("Estimated Joint PDP k=5",  woodelf_pdp_joint, dict(k = 5,         accurate = False,  centered = False, GPU = False)),
    ]

def messure_woodelf_pdp_times(model, data, params):
    running_results = {}
    for name, woodelf_func, running_params in params:
        print(name + ":")
        start_time = time.time()
        woodelf_func(model, data, **running_params)
        running_results[name] = time.time() - start_time
        print()
    return running_results

def messure_woodelf_pdp_times_for_many_depths(models, data, params, data_name):
    results = {}
    for depth, model in models.items():
        running_results = messure_woodelf_pdp_times(model, data, params)
        results[depth] = running_results
        info = {}
        info['Dataset'] = [data_name] * len(running_results)
        info['Task'] = list(running_results.keys())
        for depth, r in results.items():
            info[f'Depth {depth}'] = [round(v, 1) for v in r.values()]
        df_to_latex(pd.DataFrame(info))
    return results

In [ ]:
results = messure_woodelf_pdp_times_for_many_depths(models, data=fraud_train, params=get_pdp_testing_params(), data_name="IEEE-CIS")

Exact PDP k=5:
Building the points took: 2.3834047317504883 sec


Preprocessing the trees: 100%|██████████| 100/100 [00:00<00:00, 487.61it/s]


cache misses: 1, cache used: 199


Computing the values: 100%|██████████| 100/100 [00:00<00:00, 6735.35it/s]



Exact PDP k=100:
Building the points took: 2.358513832092285 sec


...


Estimated Joint PDP k=5:
Building the points took: 2.4287374019622803 sec. The size of the created df 225


Preprocessing the trees: 100%|██████████| 100/100 [03:37<00:00,  2.18s/it]


cache misses: 1420, cache used: 44974


Computing the values: 100%|██████████| 100/100 [01:40<00:00,  1.00s/it]



\begin{tabularx}{\columnwidth}{r|r|r|r|r|r|r|r|r|r|r|r} 
\textbf{Dataset} & \textbf{Task} & \textbf{Depth 1} & \textbf{Depth 2} & \textbf{Depth 3} & \textbf{Depth 4} & \textbf{Depth 5} & \textbf{Depth 6} & \textbf{Depth 7} & \textbf{Depth 8} & \textbf{Depth 9} & \textbf{Depth 10}\\\hline 
 IEEE-CIS  &  Exact PDP k=5  &  3.3  &  3.6  &  4.3  &  5.4  &  7.2  &  11.1  &  17.7  &  31.0  &  60.2  &  142.1  \\ 
 IEEE-CIS  &  Exact PDP k=100  &  3.4  &  3.7  &  4.2  &  5.4  &  7.2  &  10.8  &  17.6  &  30.1  &  60.5  &  142.7  \\ 
 IEEE-CIS  &  Exact full PDP  &  0.9  &  1.3  &  2.0  &  3.1  &  5.0  &  8.5  &  15.7  &  27.9  &  58.9  &  141.2  \\ 
 IEEE-CIS  &  Exact Joint PDP k=5  &  4.0  &  4.4  &  5.2  &  6.3  &  8.6  &  12.9  &  23.3  &  47.8  &  135.1  &  445.7  \\ 
 IEEE-CIS  &  Estimated PDP k=5  &  3.2  &  3.7  &  4.3  &  5.4  &  7.3  &  10.7  &  17.9  &  29.8  &  60.7  &  144.0  \\ 
 IEEE-CIS  &  Estimated Joint PDP k=5  &  4.0  &  4.1  &  4.2  &  4.6  &  5.1  &  6.5  &  11.0  &  24

In [ ]:
del transactions_test, transactions_train, fraud_test, fraud_train

# KDD-Cup 1999: Intrusion Detection Dataset

In [ ]:
detection_data = pd.read_parquet("drive/MyDrive/PATH_TO_DATA_CREATED_IN_THE_Preprocess_Data_NOTEBOOK.parquet")
unlabeled_data = pd.read_parquet("drive/MyDrive/PATH_TO_DATA_CREATED_IN_THE_Preprocess_Data_NOTEBOOK.parquet")
small_test_data = pd.read_parquet("drive/MyDrive/PATH_TO_DATA_CREATED_IN_THE_Preprocess_Data_NOTEBOOK.parquet")

detection_train_features_names = [f for f in detection_data.columns if f != "target"]
detection_trainset = detection_data[detection_train_features_names]

In [ ]:
detection_models = {depth: train_hist_gradient_boosting_model(detection_trainset, detection_data['target'], depth) for depth in tqdm(range(1, 11))}

In [ ]:
for depth, model in detection_models.items():
    y_pred = model.predict(small_test_data[detection_train_features_names])
    print(f"Depth {depth}. Accuracy: {accuracy_score(small_test_data['target'], (y_pred > 0.5).astype(int))}, F1 score: {f1_score(small_test_data['target'], (y_pred > 0.5).astype(int))}")

Depth 1. Accuracy: 0.9140208790820149, F1 score: 0.9439458029571932
Depth 2. Accuracy: 0.9235183857453807, F1 score: 0.9501801122560107
Depth 3. Accuracy: 0.9241324763928765, F1 score: 0.9506096093267611
Depth 4. Accuracy: 0.9254185301049098, F1 score: 0.9514943552619748
Depth 5. Accuracy: 0.9267753167711049, F1 score: 0.9524181602802889
Depth 6. Accuracy: 0.9264120065974556, F1 score: 0.9521739857240769
Depth 7. Accuracy: 0.92774628732369, F1 score: 0.953082038059647
Depth 8. Accuracy: 0.925659665175916, F1 score: 0.9516638166394207
Depth 9. Accuracy: 0.9261676563921692, F1 score: 0.9520201914679348
Depth 10. Accuracy: 0.9257689797414389, F1 score: 0.9517647402925704


In [ ]:
del detection_data, unlabeled_data, small_test_data

## Woodelf PDP computation


In [ ]:
results = messure_woodelf_pdp_times_for_many_depths(detection_models, data=detection_trainset, params=get_pdp_testing_params(), data_name="KDD Cup")

Exact PDP k=5:
Building the points took: 3.8407604694366455 sec


Preprocessing the trees: 100%|██████████| 100/100 [00:02<00:00, 47.66it/s]


cache misses: 1, cache used: 199


Computing the values: 100%|██████████| 100/100 [00:00<00:00, 6609.47it/s]



Exact PDP k=100:
Building the points took: 3.7561562061309814 sec


...


Estimated Joint PDP k=5:
Building the points took: 3.9806039333343506 sec. The size of the created df 175


Preprocessing the trees: 100%|██████████| 100/100 [04:08<00:00,  2.48s/it]


cache misses: 1668, cache used: 22484


Computing the values: 100%|██████████| 100/100 [00:36<00:00,  2.75it/s]



\begin{tabularx}{\columnwidth}{r|r|r|r|r|r|r|r|r|r|r|r} 
\textbf{Dataset} & \textbf{Task} & \textbf{Depth 1} & \textbf{Depth 2} & \textbf{Depth 3} & \textbf{Depth 4} & \textbf{Depth 5} & \textbf{Depth 6} & \textbf{Depth 7} & \textbf{Depth 8} & \textbf{Depth 9} & \textbf{Depth 10}\\\hline 
 KDD Cup  &  Exact PDP k=5  &  10.3  &  13.7  &  19.9  &  29.5  &  55.0  &  81.8  &  120.8  &  182.2  &  312.6  &  502.5  \\ 
 KDD Cup  &  Exact PDP k=100  &  10.6  &  13.6  &  19.9  &  29.5  &  54.0  &  81.6  &  120.0  &  182.1  &  307.9  &  499.3  \\ 
 KDD Cup  &  Exact full PDP  &  6.4  &  9.9  &  15.7  &  25.5  &  49.8  &  77.7  &  116.4  &  179.2  &  303.7  &  495.8  \\ 
 KDD Cup  &  Exact Joint PDP k=5  &  10.3  &  13.8  &  19.9  &  29.3  &  53.9  &  84.1  &  124.6  &  194.7  &  358.9  &  709.9  \\ 
 KDD Cup  &  Estimated PDP k=5  &  10.3  &  13.8  &  19.6  &  29.4  &  53.7  &  82.0  &  121.6  &  181.9  &  300.4  &  515.5  \\ 
 KDD Cup  &  Estimated Joint PDP k=5  &  8.3  &  9.6  &  11.2  &  12